In [1]:
import sys
import os

# Go up one level to the main project directory and add it to Python's path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [2]:
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

In [3]:
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from src.preprocess import preprocess_data

In [4]:
print("Loading and preprocessing training data...")
train_data = pd.read_csv(r'../data/raw/train.csv')
df = preprocess_data(train_data)

X = df.drop('health_condition', axis=1)
y = df['health_condition']

Loading and preprocessing training data...


In [5]:
print("Starting Optuna Hyperparameter Tuning with Sample Weights...")

X_train_local, X_test_local, y_train_local, y_test_local = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
def objective(trial):
    # 2. Define the hyperparameter search space
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
        'random_state': 42,
        'n_jobs': -1,
        'eval_metric': 'mlogloss'
    }

    model = XGBClassifier(**param)
    
    # 4. Calculate sample weights for the local training data ONLY

    model.fit(X_train_local, y_train_local)
    preds = model.predict(X_test_local)
    macro_f1 = f1_score(y_test_local, preds, average='macro')
    return macro_f1
    
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("\n--- Tuning Complete ---")
print(f"Best Macro F1-Score: {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")

Starting Optuna Hyperparameter Tuning with Sample Weights...


[I 2026-07-30 10:23:14,267] A new study created in memory with name: no-name-07d42a4f-3c35-49b9-b165-e10b2e598bff
[I 2026-07-30 10:23:29,931] Trial 0 finished with value: 0.9010717949798189 and parameters: {'n_estimators': 238, 'learning_rate': 0.011531001704304718, 'max_depth': 10, 'subsample': 0.8794732687994193, 'colsample_bytree': 0.8771932925588017, 'min_child_weight': 3}. Best is trial 0 with value: 0.9010717949798189.
[I 2026-07-30 10:23:44,695] Trial 1 finished with value: 0.9020210608529918 and parameters: {'n_estimators': 244, 'learning_rate': 0.014257829836623577, 'max_depth': 9, 'subsample': 0.7788811933774499, 'colsample_bytree': 0.9247846486009232, 'min_child_weight': 5}. Best is trial 1 with value: 0.9020210608529918.
[I 2026-07-30 10:24:10,364] Trial 2 finished with value: 0.9032705509456106 and parameters: {'n_estimators': 480, 'learning_rate': 0.05068837088032619, 'max_depth': 6, 'subsample': 0.6401572515934115, 'colsample_bytree': 0.718610056189017, 'min_child_weight


--- Tuning Complete ---
Best Macro F1-Score: 0.9038
Best Hyperparameters:
    n_estimators: 433
    learning_rate: 0.0636098015959294
    max_depth: 7
    subsample: 0.6816698852045894
    colsample_bytree: 0.7744659570398912
    min_child_weight: 6


In [6]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, f1_score

print("--- Local Validation: XGBoost with Prior Correction ---")

# 1. Instantiate the XGBoost model WITHOUT any class balancing
xgb_model = XGBClassifier(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

# 2. Train the unweighted model
print("Training unweighted XGBoost...")
xgb_model.fit(X_train_local, y_train_local)

# 3. Get the RAW probabilities instead of hard predictions
raw_probs = xgb_model.predict_proba(X_test_local)

# 4. Calculate the Prior Probabilities from the training set
priors = np.bincount(y_train_local) / len(y_train_local)
print(f"Training Class Priors: {priors}")

# 5. Apply the Prior Correction formula
adjusted_probs = raw_probs / priors

# 6. Make final predictions using argmax
corrected_preds = np.argmax(adjusted_probs, axis=1)

# 7. Evaluate the corrected predictions
print("\nXGBoost Local Validation Metrics (Prior Corrected):")
print(classification_report(y_test_local, corrected_preds))

macro_f1 = f1_score(y_test_local, corrected_preds, average='macro')
print(f"Corrected XGBoost Validation Macro F1-Score: {macro_f1:.4f}\n")

--- Local Validation: XGBoost with Prior Correction ---
Training unweighted XGBoost...
Training Class Priors: [0.85867553 0.05767747 0.083647  ]

XGBoost Local Validation Metrics (Prior Corrected):
              precision    recall  f1-score   support

           0       0.99      0.88      0.93    118512
           1       0.52      0.92      0.67      7961
           2       0.58      0.93      0.72     11545

    accuracy                           0.89    138018
   macro avg       0.70      0.91      0.77    138018
weighted avg       0.93      0.89      0.90    138018

Corrected XGBoost Validation Macro F1-Score: 0.7711



In [7]:
print("Retraining final model with optimized parameters...")
xgb_model = XGBClassifier(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X, y)


print("Processing Kaggle test data...")
raw_test_df = pd.read_csv(r'../data/raw/test.csv')
passenger_ids = raw_test_df['id']

clean_test_df = preprocess_data(raw_test_df)
X_test_kaggle = clean_test_df.reindex(columns=X.columns, fill_value=0)

# 4. Generate Predictions
print("Generating final predictions...")
kaggle_preds = xgb_model.predict(X_test_kaggle)

# 5. Format and Save Submission
submission = pd.DataFrame({
    'id': passenger_ids,
    'health_condition': kaggle_preds
})

# Map numeric predictions back to text labels for Kaggle
reverse_mapping = {0: 'at-risk', 1: 'fit', 2: 'unhealthy'}
submission['health_condition'] = submission['health_condition'].map(reverse_mapping)

submission.to_csv('../results/submission_12.csv', index=False)
print("Success! submission.csv is ready for Kaggle upload.")

Retraining final model with optimized parameters...
Processing Kaggle test data...
Generating final predictions...
Success! submission.csv is ready for Kaggle upload.


## Detailed Parameters

In [12]:
import optuna
import numpy as np
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.metrics import f1_score

def objective(trial):
    # 1. Define tight search spaces around the known optimal values
    params = {
        "objective": "multi:softprob",
        "num_class": 3,
        "tree_method": "hist",
        "device": "cuda", # Set to "cpu" if you don't have a GPU
        "random_state": 42,
        "n_estimators": 10000,
        "eval_metric": "mlogloss",
        
        # Micro-tuning ranges:
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.05),
        "max_depth": trial.suggest_int("max_depth", 5, 9),
        "min_child_weight": trial.suggest_float("min_child_weight", 0.7, 1.2),
        "subsample": trial.suggest_float("subsample", 0.75, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 0.65),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.4, 0.7),
        "reg_lambda": trial.suggest_float("reg_lambda", 10.0, 15.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.1, 0.5),
        "gamma": trial.suggest_float("gamma", 2.0, 5.0),
        "max_bin": trial.suggest_int("max_bin", 512, 1024),
    }

    # 2. Instantiate model with early stopping
    dtrain = xgb.DMatrix(X_train_local, label=y_train_local)
    dval = xgb.DMatrix(X_test_local, label=y_test_local)

    # 2. Add 'eval_metric' to params because we are using xgb.train() instead of the sklearn wrapper
    params["eval_metric"] = "mlogloss"

    # 3. Train using XGBoost's native API instead of the sklearn wrapper
    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=1500, # Replaces n_estimators
        evals=[(dval, 'Validation')],
        early_stopping_rounds=50,
        verbose_eval=False
    )

    # 4. Predict raw probabilities on the validation set
    raw_probs = model.predict(dval)

    # 5. Apply Prior Correction IN THE LOOP
    priors = np.bincount(y_train_local) / len(y_train_local)
    adjusted_probs = raw_probs / priors
    corrected_preds = np.argmax(adjusted_probs, axis=1)

    # 6. Calculate the final score
    macro_f1 = f1_score(y_test_local, corrected_preds, average='macro')
    
    return macro_f1

print("Starting XGBoost Micro-Tuning...")
# Direction is 'maximize' because we are returning the Macro F1 Score
study_xgb = optuna.create_study(direction="maximize", study_name="xgb_prior_correction")

# 50 trials should be plenty since the search space is so narrow
study_xgb.optimize(objective, n_trials=50)

print("\nBest XGBoost Parameters for Prior Correction:")
print(study_xgb.best_params)
print(f"Best Validation Macro F1: {study_xgb.best_value:.4f}")

[I 2026-07-30 11:08:50,358] A new study created in memory with name: xgb_prior_correction


Starting XGBoost Micro-Tuning...


[I 2026-07-30 11:09:10,263] Trial 0 finished with value: 0.7443573749958792 and parameters: {'learning_rate': 0.02080456468122314, 'max_depth': 6, 'min_child_weight': 1.0618632558266816, 'subsample': 0.9152474767125618, 'colsample_bytree': 0.5673226433757846, 'colsample_bylevel': 0.576237605847964, 'reg_lambda': 14.29010740494827, 'reg_alpha': 0.4799288987367022, 'gamma': 4.68771321797983, 'max_bin': 908}. Best is trial 0 with value: 0.7443573749958792.
[I 2026-07-30 11:09:28,573] Trial 1 finished with value: 0.7429982223505337 and parameters: {'learning_rate': 0.023893849550196425, 'max_depth': 5, 'min_child_weight': 0.7895851693646967, 'subsample': 0.7518963796181956, 'colsample_bytree': 0.5838376250266155, 'colsample_bylevel': 0.6078498861784445, 'reg_lambda': 11.541157441808064, 'reg_alpha': 0.4027507666243004, 'gamma': 4.571467212526751, 'max_bin': 832}. Best is trial 0 with value: 0.7443573749958792.
[I 2026-07-30 11:09:46,026] Trial 2 finished with value: 0.7438600250896279 and 


Best XGBoost Parameters for Prior Correction:
{'learning_rate': 0.0342770515716504, 'max_depth': 9, 'min_child_weight': 1.0568149508752573, 'subsample': 0.7787975587878133, 'colsample_bytree': 0.5945764219217395, 'colsample_bylevel': 0.6613983329633784, 'reg_lambda': 13.66505547031626, 'reg_alpha': 0.42661759861617665, 'gamma': 2.303888742852092, 'max_bin': 970}
Best Validation Macro F1: 0.7581


In [13]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier

print("Retraining final model with optimized parameters...")
xgb_model = XGBClassifier(
    **study_xgb.best_params,
    n_estimators=1500,       # <-- Added
    tree_method="hist",      # <-- Added
    device="cuda",           # <-- Added to keep it fast
    random_state=42,
    n_jobs=-1
)

# 1. Train on the full dataset WITHOUT sample weights
xgb_model.fit(X, y)

print("Processing Kaggle test data...")
raw_test_df = pd.read_csv(r'../data/raw/test.csv')
passenger_ids = raw_test_df['id']

clean_test_df = preprocess_data(raw_test_df)
X_test_kaggle = clean_test_df.reindex(columns=X.columns, fill_value=0)

# ==========================================
# 2. PRIOR CORRECTION LOGIC
# ==========================================
print("Generating final predictions with Prior Correction...")

# A. Get raw probabilities instead of hard predictions
raw_test_probs = xgb_model.predict_proba(X_test_kaggle)

# B. Calculate priors from the FULL training target (y)
priors = np.bincount(y) / len(y)
print(f"Full Dataset Priors applied: {priors}")

# C. Apply the correction formula
adjusted_test_probs = raw_test_probs / priors

# D. Extract the final predicted classes
kaggle_preds = np.argmax(adjusted_test_probs, axis=1)
# ==========================================

# 3. Format and Save Submission
submission = pd.DataFrame({
    'id': passenger_ids,
    'health_condition': kaggle_preds
})

# Map numeric predictions back to text labels for Kaggle
reverse_mapping = {0: 'at-risk', 1: 'fit', 2: 'unhealthy'}
submission['health_condition'] = submission['health_condition'].map(reverse_mapping)

submission.to_csv('../results/submission_13.csv', index=False)
print("Success! submission_13.csv is ready for Kaggle upload.")

Retraining final model with optimized parameters...
Processing Kaggle test data...
Generating final predictions with Prior Correction...
Full Dataset Priors applied: [0.85867455 0.05767815 0.0836473 ]
Success! submission_13.csv is ready for Kaggle upload.
